# Hidden Visual Evidence Master Study

This notebook consolidates the remaining experiments for:

**Self Verification Through Hidden Visual Evidence in Vision Language Models**

It is designed as one resumable RunPod workflow.

## Central question

Can a vision language model identify the regions that causally drive an answer, and does that imply that the answer is justified?

## What this notebook does

1. Loads the completed Qwen 7B and Qwen 3B POPE result bundles.
2. Measures whether causal localization transfers across the two models.
3. Compares hidden state ranking against:
   - output probability drop
   - CLIP target similarity
   - center bias
   - random ranking
   - target oracle
4. Builds a controlled COCO causal evidence benchmark from 100 unused POPE images with exact segmentation masks.
5. Runs Qwen2.5 VL 7B and 3B on the same benchmark.
6. Computes patch heatmaps and target localization metrics.
7. Measures necessity and sufficiency curves for k equal to 1, 3, and 5.
8. Separates causal localization from hallucination verification.
9. Generates a blind hallucination mechanism audit with provisional categories:
   - localized visual confusion
   - diffuse scene prior
   - semantic or annotation ambiguity candidate
   - unexplained
10. Produces paper tables, figures, LaTeX macros, a results summary, and one final zip.

## Human step

The notebook creates `taxonomy_labels.csv` and an HTML audit. The `human_label` column is intentionally blank. A human should label those examples before the taxonomy is treated as a formal paper result. Every other stage is automatic.

## Recommended RunPod

- L40S 48 GB or A100
- at least 100 GB volume storage
- expected runtime depends on caching, usually a few hours
- every expensive phase checkpoints to CSV files

## Upload to `/workspace`

Required:

- `pope_confidence_category_matched_RESULTS_20260718_062034.zip`
- `pope_crossmodel_3b_sufficiency_confirmation_RESULTS.zip`
- `pope_hidden_evidence_semantic_verification_RESULTS.zip`

Optional:

- `pope_semantic_sufficiency_confirmation_RESULTS.zip`


In [ ]:
# Cell 1: install dependencies
# Run once on a fresh pod, restart the kernel, then continue with Cell 2.

import sys
import subprocess

packages = [
    "numpy==1.26.4",
    "scipy==1.11.4",
    "pandas==2.2.2",
    "pillow==10.4.0",
    "matplotlib==3.9.2",
    "scikit-learn==1.4.2",
    "transformers==4.50.3",
    "datasets==2.19.2",
    "huggingface_hub==0.34.4",
    "accelerate==0.34.2",
    "sentencepiece",
    "requests",
    "tqdm",
    "pycocotools",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--upgrade", "pip", "setuptools", "wheel"
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--upgrade", "--no-cache-dir", *packages
])

print("Install complete. Restart the kernel, then run Cell 2.")


In [ ]:
# Cell 2: global setup, input extraction, POPE, COCO annotations, and shared helpers

import os
import re
import gc
import io
import json
import math
import random
import zipfile
import shutil
import html
import base64
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
from PIL import Image, ImageFilter, ImageDraw, ImageOps
from tqdm.auto import tqdm
from datasets import load_dataset
from pycocotools.coco import COCO
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    accuracy_score,
)
from scipy.stats import wilcoxon

# ============================================================
# Configuration
# ============================================================

WORKSPACE = Path("/workspace")
PROJECT_DIR = WORKSPACE / "pope_hidden_evidence_master_study"
INPUT_DIR = PROJECT_DIR / "input"
OUT_DIR = PROJECT_DIR / "outputs"
CACHE_DIR = PROJECT_DIR / "cache"
FIG_DIR = OUT_DIR / "figures"
AUDIT_DIR = OUT_DIR / "taxonomy_audit"

for directory in [
    PROJECT_DIR, INPUT_DIR, OUT_DIR, CACHE_DIR, FIG_DIR, AUDIT_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATASET_ID = "lmms-lab/POPE"
MODEL_SPECS = [
    {
        "tag": "qwen7b",
        "model_id": "Qwen/Qwen2.5-VL-7B-Instruct",
        "batch_size": 4,
    },
    {
        "tag": "qwen3b",
        "model_id": "Qwen/Qwen2.5-VL-3B-Instruct",
        "batch_size": 6,
    },
]

GRID_SIZE = 6
K_VALUES = [1, 3, 5]
MAX_IMAGE_SIDE = 384
BLUR_RADIUS = 14
HEATMAP_DEPTH_FRACTION = 0.75
N_CONTROLLED_IMAGES = 100
N_CONTROLLED_VAL = 30
N_CONTROLLED_EVAL = 70
CLIP_BATCH_SIZE = 64
BOOTSTRAP_ROUNDS = 2000

RUN_CONTROLLED_BENCHMARK = True
RUN_CLIP_BASELINE = True
RUN_TAXONOMY_AUDIT = True

print("Project:", PROJECT_DIR)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# ============================================================
# Input extraction
# ============================================================

def find_zip(patterns):
    candidates = []
    for pattern in patterns:
        candidates.extend(WORKSPACE.glob(pattern))
    candidates = [
        path for path in candidates
        if path.is_file() and zipfile.is_zipfile(path)
    ]
    return max(candidates, key=lambda path: path.stat().st_size) if candidates else None

def extract_once(name, patterns, required_file):
    destination = INPUT_DIR / name
    matches = list(destination.rglob(required_file)) if destination.exists() else []
    if matches:
        return destination

    zip_path = find_zip(patterns)
    if zip_path is None:
        raise FileNotFoundError(
            f"Could not find required result zip for {name}. "
            f"Expected a /workspace file matching: {patterns}"
        )

    print(f"Extracting {name}:", zip_path)
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(destination)

    matches = list(destination.rglob(required_file))
    if not matches:
        raise FileNotFoundError(
            f"{required_file} was not found after extracting {zip_path}"
        )
    return destination

DIR_7B = extract_once(
    "qwen7b_matched",
    [
        "*confidence_category_matched*RESULTS*.zip",
        "*confidence_category_matched*.zip",
    ],
    "selected_confidence_category_matched.csv",
)

DIR_3B = extract_once(
    "qwen3b_crossmodel",
    [
        "*crossmodel_3b_sufficiency_confirmation*RESULTS*.zip",
        "*crossmodel_3b_sufficiency_confirmation*.zip",
    ],
    "selected_crossmodel_confirmation.csv",
)

DIR_SEMANTIC = extract_once(
    "qwen7b_semantic",
    [
        "*hidden_evidence_semantic_verification*RESULTS*.zip",
        "*hidden_evidence_semantic_verification*.zip",
    ],
    "semantic_verification_eval_result.json",
)

def find_file(root, name):
    matches = list(Path(root).rglob(name))
    if not matches:
        raise FileNotFoundError(f"{name} not found under {root}")
    return matches[0]

selected_7b = pd.read_csv(
    find_file(DIR_7B, "selected_confidence_category_matched.csv")
)
grid_7b = pd.read_csv(
    find_file(DIR_7B, "grid_patch_results.csv")
)
dist_7b = pd.read_csv(
    find_file(DIR_7B, "grid_patch_layer_distances.csv")
)
tasks_7b = pd.read_csv(
    find_file(DIR_7B, "selfverification_tasks.csv")
)
conditions_7b = pd.read_csv(
    find_file(DIR_7B, "selfverification_answer_conditions.csv")
)

selected_3b = pd.read_csv(
    find_file(DIR_3B, "selected_crossmodel_confirmation.csv")
)
heat_3b = pd.read_csv(
    find_file(DIR_3B, "crossmodel_heatmap_results.csv")
)
tasks_3b = pd.read_csv(
    find_file(DIR_3B, "crossmodel_sufficiency_tasks.csv")
)
suff_3b = pd.read_csv(
    find_file(DIR_3B, "crossmodel_sufficiency_results.csv")
)
analysis_3b = pd.read_csv(
    find_file(DIR_3B, "crossmodel_analysis.csv")
)
result_3b = json.loads(
    find_file(DIR_3B, "crossmodel_confirmation_result.json").read_text()
)

semantic_all_7b = pd.read_csv(
    find_file(DIR_SEMANTIC, "semantic_verification_all_examples.csv")
)
semantic_eval_7b = pd.read_csv(
    find_file(DIR_SEMANTIC, "semantic_verification_eval_predictions.csv")
)
semantic_result_7b = json.loads(
    find_file(DIR_SEMANTIC, "semantic_verification_eval_result.json").read_text()
)

print("Loaded prior results")
print("7B selected:", len(selected_7b))
print("3B selected:", len(selected_3b))

# ============================================================
# POPE dataset
# ============================================================

dataset = load_dataset(DATASET_ID, split="test")
print("POPE rows:", len(dataset))
print("POPE columns:", dataset.column_names)

def get_field(row, names, default=None):
    for name in names:
        if name in row and row[name] is not None:
            return row[name]
    return default

def get_image(row_idx):
    row = dataset[int(row_idx)]
    image = row.get("image")
    if isinstance(image, Image.Image):
        return image.convert("RGB")
    if hasattr(image, "convert"):
        return image.convert("RGB")
    raise ValueError(f"No PIL image for row {row_idx}")

def parse_coco_id_and_split(row_or_index):
    row = dataset[int(row_or_index)] if isinstance(
        row_or_index, (int, np.integer)
    ) else row_or_index

    source = str(get_field(row, ["image_source", "source"], ""))
    match = re.search(r"COCO_(train|val)2014_(\d+)", source)
    if match:
        return match.group(1), int(match.group(2))

    value = get_field(row, ["image_id", "coco_image_id"], None)
    if value is not None:
        return "val", int(value)

    match = re.search(r"(\d{6,12})", source)
    if match:
        return "val", int(match.group(1))

    raise ValueError(f"Could not parse COCO image ID from {source!r}")

# ============================================================
# COCO annotations with SSL fallback
# ============================================================

ANN_DIR = CACHE_DIR / "coco_annotations"
ANN_DIR.mkdir(parents=True, exist_ok=True)
ANN_ZIP = ANN_DIR / "annotations_trainval2014.zip"
ANN_URL = (
    "https://images.cocodataset.org/annotations/"
    "annotations_trainval2014.zip"
)

def download_stream(url, destination, verify_ssl=True):
    partial = destination.with_suffix(destination.suffix + ".part")
    partial.unlink(missing_ok=True)
    with requests.get(
        url,
        stream=True,
        verify=verify_ssl,
        timeout=(30, 600),
        headers={"User-Agent": "Mozilla/5.0"},
    ) as response:
        response.raise_for_status()
        with partial.open("wb") as file:
            for chunk in response.iter_content(1024 * 1024):
                if chunk:
                    file.write(chunk)
    if not zipfile.is_zipfile(partial):
        partial.unlink(missing_ok=True)
        raise RuntimeError("Downloaded COCO annotations are not a valid zip")
    partial.replace(destination)

instances_val = ANN_DIR / "annotations" / "instances_val2014.json"
instances_train = ANN_DIR / "annotations" / "instances_train2014.json"

if not instances_val.exists() or not instances_train.exists():
    if ANN_ZIP.exists() and not zipfile.is_zipfile(ANN_ZIP):
        ANN_ZIP.unlink()

    if not ANN_ZIP.exists():
        print("Downloading COCO annotations")
        try:
            download_stream(ANN_URL, ANN_ZIP, verify_ssl=True)
        except requests.exceptions.SSLError:
            print("COCO SSL mismatch, retrying without certificate verification")
            download_stream(ANN_URL, ANN_ZIP, verify_ssl=False)

    print("Extracting COCO annotations")
    with zipfile.ZipFile(ANN_ZIP, "r") as archive:
        archive.extractall(ANN_DIR)

coco_val = COCO(str(instances_val))
coco_train = COCO(str(instances_train))
COCO_BY_SPLIT = {"val": coco_val, "train": coco_train}

cat_name_to_id = {}
cat_id_to_name = {}
for coco in [coco_val, coco_train]:
    for category in coco.loadCats(coco.getCatIds()):
        cat_name_to_id[category["name"]] = int(category["id"])
        cat_id_to_name[int(category["id"])] = category["name"]

COCO_CATEGORIES = sorted(cat_name_to_id)

# ============================================================
# Shared target and patch helpers
# ============================================================

def target_union_mask(row_idx, target_category, image_size):
    coco_split, image_id = parse_coco_id_and_split(int(row_idx))
    coco = COCO_BY_SPLIT[coco_split]
    category_id = cat_name_to_id.get(str(target_category))
    if category_id is None:
        return np.zeros(
            (image_size[1], image_size[0]), dtype=bool
        ), [], coco_split, image_id

    annotation_ids = coco.getAnnIds(
        imgIds=[int(image_id)],
        catIds=[int(category_id)],
        iscrowd=None,
    )
    annotations = coco.loadAnns(annotation_ids)
    masks = []
    boxes = []

    for annotation in annotations:
        mask = coco.annToMask(annotation).astype(bool)
        if mask.shape != (image_size[1], image_size[0]):
            mask = np.array(
                Image.fromarray(mask.astype(np.uint8) * 255).resize(
                    image_size, Image.Resampling.NEAREST
                )
            ) > 0
        masks.append(mask)
        boxes.append(annotation.get("bbox", [0, 0, 0, 0]))

    union = (
        np.any(np.stack(masks, axis=0), axis=0)
        if masks
        else np.zeros((image_size[1], image_size[0]), dtype=bool)
    )
    return union, boxes, coco_split, int(image_id)

def patch_boxes(width, height, grid=GRID_SIZE):
    boxes = []
    for grid_y in range(grid):
        for grid_x in range(grid):
            x1 = int(round(grid_x * width / grid))
            y1 = int(round(grid_y * height / grid))
            x2 = int(round((grid_x + 1) * width / grid))
            y2 = int(round((grid_y + 1) * height / grid))
            boxes.append((x1, y1, x2, y2, grid_x, grid_y))
    return boxes

def patch_target_table(image, target_mask):
    target_area = float(target_mask.sum())
    rows = []
    for patch_index, (
        x1, y1, x2, y2, grid_x, grid_y
    ) in enumerate(patch_boxes(image.width, image.height)):
        patch_mask = target_mask[y1:y2, x1:x2]
        overlap = float(patch_mask.sum())
        patch_area = max(1.0, float((x2 - x1) * (y2 - y1)))
        rows.append({
            "patch_index": int(patch_index),
            "grid_x": int(grid_x),
            "grid_y": int(grid_y),
            "patch_box_json": json.dumps([x1, y1, x2, y2]),
            "patch_has_target": int(overlap > 0),
            "patch_target_overlap_frac": float(overlap / patch_area),
            "target_coverage_frac": float(
                overlap / target_area
            ) if target_area > 0 else 0.0,
        })
    return pd.DataFrame(rows)

def apply_patch_blur(image, boxes, radius=BLUR_RADIUS):
    image = image.convert("RGB")
    output = image.copy()
    blurred = image.filter(ImageFilter.GaussianBlur(radius=radius))
    for box in boxes:
        x1, y1, x2, y2 = [int(value) for value in box[:4]]
        output.paste(blurred.crop((x1, y1, x2, y2)), (x1, y1))
    return output

def keep_only_boxes(image, boxes, radius=BLUR_RADIUS):
    image = image.convert("RGB")
    background = image.filter(
        ImageFilter.GaussianBlur(radius=radius * 1.6)
    )
    for box in boxes:
        x1, y1, x2, y2 = [int(value) for value in box[:4]]
        background.paste(image.crop((x1, y1, x2, y2)), (x1, y1))
    return background

def normalized_entropy(scores):
    scores = np.asarray(scores, dtype=float)
    scores = np.maximum(scores, 0.0)
    if scores.sum() <= 0:
        return 1.0
    probabilities = scores / scores.sum()
    entropy = -np.sum(
        probabilities * np.log(probabilities + 1e-12)
    )
    return float(entropy / np.log(len(probabilities)))

def center_score_for_boxes(box_json_values, image_size):
    width, height = image_size
    center_x = width / 2.0
    center_y = height / 2.0
    scale = math.sqrt(width * width + height * height) + 1e-12
    scores = []
    for value in box_json_values:
        x1, y1, x2, y2 = json.loads(value)
        patch_x = (x1 + x2) / 2.0
        patch_y = (y1 + y2) / 2.0
        distance = math.sqrt(
            (patch_x - center_x) ** 2
            + (patch_y - center_y) ** 2
        )
        scores.append(-distance / scale)
    return np.asarray(scores, dtype=float)

def localization_metrics_for_group(group, score_column):
    group = group.copy()
    group[score_column] = pd.to_numeric(
        group[score_column], errors="coerce"
    ).fillna(0.0)

    y_patch = group.patch_has_target.astype(int).to_numpy()
    scores = group[score_column].to_numpy(float)
    order = np.argsort(-scores)

    first_target_rank = len(order) + 1
    for rank, index in enumerate(order, start=1):
        if y_patch[index] == 1:
            first_target_rank = rank
            break

    row = {
        "patch_ap": float(
            average_precision_score(y_patch, scores)
        ) if len(np.unique(y_patch)) == 2 else float(y_patch.mean()),
        "first_target_rank": int(first_target_rank),
        "reciprocal_first_target_rank": float(1.0 / first_target_rank),
    }

    for k in K_VALUES:
        selected_rows = group.iloc[order[:k]]
        row[f"top{k}_hit"] = int(
            selected_rows.patch_has_target.max() > 0
        )
        row[f"top{k}_coverage"] = float(
            min(1.0, selected_rows.target_coverage_frac.sum())
        )
    return row

def bootstrap_mean(values, rounds=BOOTSTRAP_ROUNDS, seed=SEED):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    means = []
    for _ in range(rounds):
        sample = values[
            rng.integers(0, len(values), size=len(values))
        ]
        means.append(float(np.mean(sample)))
    return {
        "mean": float(np.mean(values)),
        "ci_low": float(np.quantile(means, 0.025)),
        "ci_high": float(np.quantile(means, 0.975)),
    }

def image_to_base64(image):
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=90)
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

print("Cell 2 complete.")


In [ ]:
# Cell 3: existing cross-model localization transfer analysis

from sklearn.metrics import average_precision_score

LOCALIZATION_DIR = OUT_DIR / "existing_localization"
LOCALIZATION_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Qwen 7B existing grounded evaluation examples
# ============================================================

HEATMAP_LAYER_7B = 21

dist_layer_7b = dist_7b[
    pd.to_numeric(dist_7b["layer"], errors="coerce").astype(int)
    == HEATMAP_LAYER_7B
].copy()

patch_7b = grid_7b[[
    "selected_index",
    "patch_index",
    "patch_box_json",
    "patch_has_target",
    "patch_target_overlap_frac",
    "target_coverage_frac",
    "p_yes_drop",
]].drop_duplicates(["selected_index", "patch_index"])

loc_7b = dist_layer_7b.merge(
    patch_7b,
    on=["selected_index", "patch_index"],
    how="left",
    suffixes=("", "_grid"),
)
loc_7b = loc_7b.merge(
    selected_7b[[
        "selected_index",
        "row_idx",
        "split",
        "clean_group",
        "question",
        "target_category",
        "clean_p_yes",
    ]],
    on="selected_index",
    how="left",
    suffixes=("", "_selected"),
)

loc_7b = loc_7b[
    (loc_7b.split_selected.astype(str) == "eval")
    & (loc_7b.clean_group_selected.astype(str) == "grounded_yes")
].copy()

# ============================================================
# Qwen 3B existing grounded evaluation examples
# ============================================================

selected_3b_eval = selected_3b[
    (selected_3b.split.astype(str) == "eval")
    & (selected_3b.clean_group.astype(str) == "grounded_yes")
].copy()

target_tables_3b = []

for _, meta in tqdm(
    selected_3b_eval.iterrows(),
    total=len(selected_3b_eval),
    desc="3B target overlap reconstruction",
):
    image = get_image(int(meta.row_idx))
    mask, _, _, _ = target_union_mask(
        int(meta.row_idx),
        str(meta.target_category),
        image.size,
    )
    target_table = patch_target_table(image, mask)
    target_table["selected_index"] = int(meta.selected_index)
    target_tables_3b.append(target_table)

target_patch_3b = pd.concat(target_tables_3b, ignore_index=True)

loc_3b = heat_3b.merge(
    target_patch_3b,
    on=["selected_index", "patch_index"],
    how="inner",
)
loc_3b = loc_3b.merge(
    selected_3b[[
        "selected_index",
        "row_idx",
        "split",
        "clean_group",
        "question",
        "target_category",
        "clean_p_yes",
    ]],
    on="selected_index",
    how="left",
    suffixes=("", "_selected"),
)

loc_3b["p_yes_drop"] = (
    pd.to_numeric(loc_3b.clean_p_yes, errors="coerce")
    - pd.to_numeric(loc_3b.perturb_p_yes, errors="coerce")
)

# ============================================================
# Add center and deterministic random rankings
# ============================================================

def add_non_model_scores(frame):
    frame = frame.copy()

    # Merges can create suffixed duplicates such as
    # patch_box_json_x and patch_box_json_y.
    if "patch_box_json" not in frame.columns:
        patch_box_candidates = [
            column
            for column in [
                "patch_box_json_x",
                "patch_box_json_y",
                "patch_box_json_grid",
                "patch_box_json_base",
            ]
            if column in frame.columns
        ]
        if not patch_box_candidates:
            raise RuntimeError(
                "No patch box column was found. Available columns: "
                + ", ".join(frame.columns.astype(str))
            )
        frame["patch_box_json"] = frame[
            patch_box_candidates[0]
        ]

    if "patch_target_overlap_frac" not in frame.columns:
        overlap_candidates = [
            column
            for column in [
                "patch_target_overlap_frac_x",
                "patch_target_overlap_frac_y",
                "patch_target_overlap_frac_grid",
                "patch_target_overlap_frac_base",
            ]
            if column in frame.columns
        ]
        if not overlap_candidates:
            raise RuntimeError(
                "No target-overlap column was found."
            )
        frame["patch_target_overlap_frac"] = frame[
            overlap_candidates[0]
        ]

    parts = []
    for selected_index, group in frame.groupby(
        "selected_index", sort=False
    ):
        group = group.copy().sort_values("patch_index")
        meta = group.iloc[0]
        image = get_image(int(meta.row_idx))
        group["center_score"] = center_score_for_boxes(
            group["patch_box_json"].astype(str).tolist(),
            image.size,
        )
        rng = np.random.default_rng(
            SEED + int(selected_index) * 193
        )
        group["random_score"] = rng.random(len(group))
        group["oracle_score"] = (
            pd.to_numeric(
                group["patch_target_overlap_frac"],
                errors="coerce",
            ).fillna(0.0)
        )
        parts.append(group)
    return pd.concat(parts, ignore_index=True)

loc_7b = add_non_model_scores(loc_7b)
loc_3b = add_non_model_scores(loc_3b)

# ============================================================
# Metrics
# ============================================================

method_columns = {
    "hidden_state": "relative_l2",
    "output_drop": "p_yes_drop",
    "center": "center_score",
    "random": "random_score",
    "target_oracle": "oracle_score",
}

metric_rows = []
per_image_rows = []

for model_tag, frame in [
    ("qwen7b", loc_7b),
    ("qwen3b", loc_3b),
]:
    for selected_index, group in frame.groupby(
        "selected_index", sort=False
    ):
        for method, column in method_columns.items():
            metrics = localization_metrics_for_group(
                group, column
            )
            record = {
                "model": model_tag,
                "selected_index": int(selected_index),
                "method": method,
                **metrics,
            }
            per_image_rows.append(record)

per_image = pd.DataFrame(per_image_rows)

for (model_tag, method), group in per_image.groupby(
    ["model", "method"], sort=False
):
    row = {
        "model": model_tag,
        "method": method,
        "n_images": int(len(group)),
    }
    for column in [
        "patch_ap",
        "reciprocal_first_target_rank",
        "top1_hit",
        "top1_coverage",
        "top3_hit",
        "top3_coverage",
        "top5_hit",
        "top5_coverage",
    ]:
        summary = bootstrap_mean(
            group[column].to_numpy(float),
            seed=SEED + abs(hash((model_tag, method, column))) % 10000,
        )
        row[column] = summary["mean"]
        row[f"{column}_ci_low"] = summary["ci_low"]
        row[f"{column}_ci_high"] = summary["ci_high"]
    metric_rows.append(row)

existing_localization_summary = pd.DataFrame(metric_rows)
existing_localization_summary.to_csv(
    LOCALIZATION_DIR / "existing_crossmodel_localization_summary.csv",
    index=False,
)
per_image.to_csv(
    LOCALIZATION_DIR / "existing_crossmodel_localization_per_image.csv",
    index=False,
)

print(existing_localization_summary[[
    "model",
    "method",
    "patch_ap",
    "top1_hit",
    "top3_coverage",
    "top5_coverage",
]])

print("Cell 3 complete.")


In [ ]:
# Cell 4: build the controlled COCO causal evidence benchmark and CLIP patch baseline

CONTROL_DIR = OUT_DIR / "controlled_benchmark"
CONTROL_CACHE = CACHE_DIR / "controlled_benchmark"
CONTROL_DIR.mkdir(parents=True, exist_ok=True)
CONTROL_CACHE.mkdir(parents=True, exist_ok=True)

benchmark_csv = CONTROL_DIR / "controlled_coco_evidence_benchmark.csv"
patch_csv = CONTROL_DIR / "controlled_coco_patch_metadata.csv"

# Exclude all images used by prior matched experiments.
excluded_image_ids = set(
    pd.to_numeric(
        selected_7b["image_id"], errors="coerce"
    ).dropna().astype(int).tolist()
)
excluded_image_ids.update(
    pd.to_numeric(
        selected_3b["image_id"], errors="coerce"
    ).dropna().astype(int).tolist()
)

def category_masks_for_image(row_idx, image_size):
    coco_split, image_id = parse_coco_id_and_split(int(row_idx))
    coco = COCO_BY_SPLIT[coco_split]
    annotation_ids = coco.getAnnIds(
        imgIds=[int(image_id)],
        iscrowd=None,
    )
    annotations = coco.loadAnns(annotation_ids)

    by_category = {}
    for annotation in annotations:
        category_name = cat_id_to_name.get(
            int(annotation["category_id"])
        )
        if category_name is None:
            continue
        mask = coco.annToMask(annotation).astype(bool)
        if mask.shape != (image_size[1], image_size[0]):
            mask = np.array(
                Image.fromarray(mask.astype(np.uint8) * 255).resize(
                    image_size, Image.Resampling.NEAREST
                )
            ) > 0
        by_category.setdefault(category_name, []).append(mask)

    unions = {
        category: np.any(np.stack(masks, axis=0), axis=0)
        for category, masks in by_category.items()
    }
    return unions, coco_split, int(image_id)

if benchmark_csv.exists() and patch_csv.exists():
    benchmark = pd.read_csv(benchmark_csv)
    benchmark_patches = pd.read_csv(patch_csv)
else:
    unique_candidates = []
    seen_images = set()

    for row_idx in tqdm(
        range(len(dataset)),
        desc="controlled COCO benchmark candidates",
    ):
        row = dataset[int(row_idx)]
        try:
            coco_split, image_id = parse_coco_id_and_split(row)
        except Exception:
            continue

        if image_id in seen_images or image_id in excluded_image_ids:
            continue

        image = get_image(row_idx)
        unions, coco_split, image_id = category_masks_for_image(
            row_idx, image.size
        )
        if not unions:
            continue

        category_areas = {
            category: float(mask.mean())
            for category, mask in unions.items()
        }

        eligible = [
            (category, area)
            for category, area in category_areas.items()
            if 0.008 <= area <= 0.45
        ]
        if not eligible:
            continue

        non_person = [
            item for item in eligible
            if item[0] != "person"
        ]
        pool = non_person if non_person else eligible
        target_category, target_area = max(
            pool, key=lambda item: item[1]
        )

        unique_candidates.append({
            "row_idx": int(row_idx),
            "image_id": int(image_id),
            "coco_split": coco_split,
            "target_category": target_category,
            "target_area_frac": float(target_area),
            "question": (
                f"Is there a {target_category} in the image?"
            ),
        })
        seen_images.add(image_id)

        if len(unique_candidates) >= N_CONTROLLED_IMAGES * 3:
            break

    candidate_frame = pd.DataFrame(unique_candidates)
    candidate_frame = candidate_frame.sample(
        frac=1.0, random_state=SEED
    ).reset_index(drop=True)

    if len(candidate_frame) < N_CONTROLLED_IMAGES:
        raise RuntimeError(
            f"Only found {len(candidate_frame)} controlled images, "
            f"need {N_CONTROLLED_IMAGES}"
        )

    benchmark = candidate_frame.iloc[
        :N_CONTROLLED_IMAGES
    ].copy()
    benchmark["benchmark_index"] = np.arange(len(benchmark))
    benchmark["split"] = (
        ["val"] * N_CONTROLLED_VAL
        + ["eval"] * N_CONTROLLED_EVAL
    )
    benchmark.to_csv(benchmark_csv, index=False)

    patch_parts = []
    for _, meta in tqdm(
        benchmark.iterrows(),
        total=len(benchmark),
        desc="controlled target patch metadata",
    ):
        image = get_image(int(meta.row_idx))
        target_mask, _, _, _ = target_union_mask(
            int(meta.row_idx),
            str(meta.target_category),
            image.size,
        )
        table = patch_target_table(image, target_mask)
        table["benchmark_index"] = int(meta.benchmark_index)
        table["row_idx"] = int(meta.row_idx)
        table["image_id"] = int(meta.image_id)
        table["target_category"] = str(meta.target_category)
        patch_parts.append(table)

    benchmark_patches = pd.concat(
        patch_parts, ignore_index=True
    )
    benchmark_patches.to_csv(patch_csv, index=False)

print("Controlled benchmark images:", len(benchmark))
print(benchmark.groupby("split").size())

# ============================================================
# CLIP target-specific patch baseline
# ============================================================

clip_scores_csv = CONTROL_DIR / "controlled_clip_patch_scores.csv"

if RUN_CLIP_BASELINE:
    if clip_scores_csv.exists():
        clip_patch_scores = pd.read_csv(clip_scores_csv)
    else:
        from transformers import CLIPModel, CLIPProcessor

        clip_model_id = "openai/clip-vit-base-patch32"
        device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )
        clip_dtype = (
            torch.float16
            if device.type == "cuda"
            else torch.float32
        )

        clip_processor = CLIPProcessor.from_pretrained(
            clip_model_id
        )
        clip_model = CLIPModel.from_pretrained(
            clip_model_id,
            torch_dtype=clip_dtype,
        ).to(device)
        clip_model.eval()

        unique_targets = sorted(
            benchmark.target_category.unique().tolist()
        )
        target_texts = [
            f"a clear photo of a {target}"
            for target in unique_targets
        ]
        text_inputs = clip_processor(
            text=target_texts,
            padding=True,
            return_tensors="pt",
        )
        text_inputs = {
            key: value.to(device)
            for key, value in text_inputs.items()
        }

        with torch.inference_mode():
            text_features = clip_model.get_text_features(
                **text_inputs
            )
            text_features = text_features / (
                text_features.norm(
                    dim=-1, keepdim=True
                ) + 1e-12
            )

        target_to_feature = {
            target: text_features[index]
            for index, target in enumerate(unique_targets)
        }

        tasks = []
        for _, patch in benchmark_patches.iterrows():
            meta = benchmark[
                benchmark.benchmark_index
                == int(patch.benchmark_index)
            ].iloc[0]
            image = get_image(int(meta.row_idx))
            box = [
                int(value)
                for value in json.loads(patch.patch_box_json)
            ]
            crop = image.crop(tuple(box)).convert("RGB")
            tasks.append({
                "benchmark_index": int(patch.benchmark_index),
                "patch_index": int(patch.patch_index),
                "target_category": str(meta.target_category),
                "crop": crop,
            })

        clip_rows = []

        for start in tqdm(
            range(0, len(tasks), CLIP_BATCH_SIZE),
            desc="controlled CLIP patch baseline",
        ):
            batch = tasks[start:start + CLIP_BATCH_SIZE]
            images = [task["crop"] for task in batch]
            image_inputs = clip_processor(
                images=images,
                return_tensors="pt",
            )
            pixel_values = image_inputs["pixel_values"].to(
                device=device,
                dtype=clip_dtype,
            )

            with torch.inference_mode():
                image_features = clip_model.get_image_features(
                    pixel_values=pixel_values
                )
                image_features = image_features / (
                    image_features.norm(
                        dim=-1, keepdim=True
                    ) + 1e-12
                )

            for index, task in enumerate(batch):
                target_feature = target_to_feature[
                    task["target_category"]
                ]
                similarity = float(
                    torch.dot(
                        image_features[index],
                        target_feature,
                    ).detach().cpu()
                )
                clip_rows.append({
                    "benchmark_index": int(task["benchmark_index"]),
                    "patch_index": int(task["patch_index"]),
                    "clip_score": similarity,
                })

        clip_patch_scores = pd.DataFrame(clip_rows)
        clip_patch_scores.to_csv(
            clip_scores_csv, index=False
        )

        del clip_model, clip_processor, text_features
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    clip_patch_scores = pd.DataFrame(columns=[
        "benchmark_index", "patch_index", "clip_score"
    ])

print("Cell 4 complete.")


In [ ]:
# Cell 5: shared Qwen runner for the controlled benchmark

from transformers import AutoProcessor

try:
    from transformers import Qwen2_5_VLForConditionalGeneration as QwenVLClass
except Exception:
    from transformers import Qwen2VLForConditionalGeneration as QwenVLClass

def model_dtype():
    device_name = (
        torch.cuda.get_device_name(0).lower()
        if torch.cuda.is_available()
        else ""
    )
    use_float16 = any(
        name in device_name
        for name in ["a40", "rtx", "a5000", "a6000"]
    )
    if torch.cuda.is_available() and use_float16:
        return torch.float16
    if torch.cuda.is_available():
        return torch.bfloat16
    return torch.float32

def load_qwen(model_id):
    dtype = model_dtype()
    print("Loading:", model_id, "dtype:", dtype)

    processor = AutoProcessor.from_pretrained(
        model_id,
        trust_remote_code=True,
        use_fast=False,
    )
    if hasattr(processor, "tokenizer") and processor.tokenizer is not None:
        processor.tokenizer.padding_side = "left"

    model = QwenVLClass.from_pretrained(
        model_id,
        torch_dtype=dtype,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )
    if not torch.cuda.is_available():
        model = model.to("cpu")
    model.eval()
    return model, processor

def resize_for_vlm(image, max_side=MAX_IMAGE_SIDE):
    image = image.convert("RGB")
    width, height = image.size
    largest = max(width, height)
    if largest <= max_side:
        return image
    scale = max_side / float(largest)
    return image.resize(
        (
            max(1, int(round(width * scale))),
            max(1, int(round(height * scale))),
        ),
        Image.Resampling.LANCZOS,
    )

def collect_token_ids(tokenizer, words):
    ids = []
    for word in words:
        for variant in [
            word,
            " " + word,
            word.capitalize(),
            " " + word.capitalize(),
        ]:
            tokens = tokenizer.encode(
                variant, add_special_tokens=False
            )
            if len(tokens) == 1:
                ids.append(tokens[0])
    return sorted(set(ids))

def qwen_batch(
    model,
    processor,
    images,
    prompts,
    return_hidden=False,
):
    tokenizer = processor.tokenizer
    yes_ids = collect_token_ids(tokenizer, ["yes"])
    no_ids = collect_token_ids(tokenizer, ["no"])

    messages = [
        [{
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }]
        for prompt in prompts
    ]
    texts = [
        processor.apply_chat_template(
            message,
            tokenize=False,
            add_generation_prompt=True,
        )
        for message in messages
    ]
    safe_images = [
        resize_for_vlm(image)
        for image in images
    ]

    inputs = processor(
        text=texts,
        images=safe_images,
        return_tensors="pt",
        padding=True,
    )
    device = next(model.parameters()).device
    inputs = {
        key: (
            value.to(device)
            if hasattr(value, "to")
            else value
        )
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            do_sample=False,
            num_beams=1,
            max_new_tokens=2,
            return_dict_in_generate=True,
            output_scores=True,
            output_hidden_states=return_hidden,
            use_cache=True,
        )

    input_length = inputs["input_ids"].shape[1]
    generated_ids = generated.sequences[:, input_length:]
    raw = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    raw = [text.strip() for text in raw]

    first_scores = generated.scores[0].float()
    probabilities = torch.softmax(first_scores, dim=-1)
    yes_probability = probabilities[:, yes_ids].sum(dim=-1)
    no_probability = probabilities[:, no_ids].sum(dim=-1)
    p_yes = (
        yes_probability
        / (yes_probability + no_probability + 1e-12)
    ).detach().cpu().numpy().astype(float)
    prediction = (p_yes >= 0.5).astype(int)

    hidden = None
    if return_hidden:
        hidden_tuple = generated.hidden_states[0]
        hidden = torch.stack(
            [
                layer_hidden[:, -1, :].detach().float().cpu()
                for layer_hidden in hidden_tuple
            ],
            dim=1,
        ).numpy().astype(np.float16)

    del inputs, generated, generated_ids, first_scores, probabilities
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "raw": raw,
        "p_yes": p_yes,
        "pred": prediction,
        "hidden": hidden,
    }

def answer_prompt(question):
    return (
        "Answer only yes or no based only on the visible image.\n"
        f"Question: {question}"
    )

def run_controlled_model(spec):
    tag = spec["tag"]
    model_id = spec["model_id"]
    batch_size = int(spec["batch_size"])

    model_dir = CONTROL_DIR / tag
    model_cache = CONTROL_CACHE / tag
    hidden_dir = model_cache / "clean_hidden"
    evidence_dir = model_cache / "evidence_images"

    for directory in [
        model_dir, model_cache, hidden_dir, evidence_dir
    ]:
        directory.mkdir(parents=True, exist_ok=True)

    model, processor = load_qwen(model_id)

    clean_csv = model_dir / "clean_predictions.csv"
    if clean_csv.exists():
        clean = pd.read_csv(clean_csv)
    else:
        clean = pd.DataFrame()

    completed = set(
        pd.to_numeric(
            clean.get("benchmark_index", pd.Series(dtype=int)),
            errors="coerce",
        ).dropna().astype(int).tolist()
    )
    pending = benchmark[
        ~benchmark.benchmark_index.astype(int).isin(completed)
    ].reset_index(drop=True)

    new_rows = []

    for start in tqdm(
        range(0, len(pending), batch_size),
        desc=f"{tag} controlled clean",
    ):
        batch = pending.iloc[start:start + batch_size]
        images = [
            get_image(int(row.row_idx))
            for _, row in batch.iterrows()
        ]
        prompts = [
            answer_prompt(row.question)
            for _, row in batch.iterrows()
        ]
        output = qwen_batch(
            model,
            processor,
            images,
            prompts,
            return_hidden=True,
        )

        for index, (_, row) in enumerate(batch.iterrows()):
            record = row.to_dict()
            record.update({
                "clean_raw": output["raw"][index],
                "clean_p_yes": float(output["p_yes"][index]),
                "clean_pred": int(output["pred"][index]),
                "clean_correct": int(output["pred"][index] == 1),
            })
            new_rows.append(record)
            np.save(
                hidden_dir
                / f"benchmark_{int(row.benchmark_index)}.npy",
                output["hidden"][index],
            )

        if (
            len(new_rows) >= 100
            or start + batch_size >= len(pending)
        ):
            clean = pd.concat(
                [clean, pd.DataFrame(new_rows)],
                ignore_index=True,
            )
            clean = clean.drop_duplicates(
                "benchmark_index", keep="last"
            )
            clean.to_csv(clean_csv, index=False)
            new_rows = []

    # Use correct yes responses for causal localization.
    selected_model = clean[
        clean.clean_pred.astype(int) == 1
    ].copy()

    if len(selected_model) < 60:
        print(
            f"Warning: {tag} has only {len(selected_model)} "
            "correct yes controlled examples"
        )

    sample_hidden = np.load(
        hidden_dir
        / f"benchmark_{int(selected_model.iloc[0].benchmark_index)}.npy"
    )
    n_hidden_states = int(sample_hidden.shape[0])
    heatmap_layer = int(
        round(
            HEATMAP_DEPTH_FRACTION
            * (n_hidden_states - 1)
        )
    )

    manifest = {
        "model_id": model_id,
        "tag": tag,
        "n_hidden_states": n_hidden_states,
        "heatmap_layer": heatmap_layer,
        "grid_size": GRID_SIZE,
        "k_values": K_VALUES,
    }
    (
        model_dir / "manifest.json"
    ).write_text(json.dumps(manifest, indent=2))

    heatmap_csv = model_dir / "heatmap_results.csv"
    if heatmap_csv.exists():
        heatmap = pd.read_csv(heatmap_csv)
    else:
        heatmap = pd.DataFrame()

    done = set(
        heatmap.get(
            "condition_id", pd.Series(dtype=str)
        ).astype(str).tolist()
    )

    task_rows = []
    patch_lookup = benchmark_patches.set_index(
        ["benchmark_index", "patch_index"]
    )

    for _, meta in selected_model.iterrows():
        for patch_index in range(GRID_SIZE * GRID_SIZE):
            condition_id = (
                f"{int(meta.benchmark_index)}|{patch_index}"
            )
            if condition_id in done:
                continue
            patch = patch_lookup.loc[
                (int(meta.benchmark_index), int(patch_index))
            ]
            task_rows.append({
                "condition_id": condition_id,
                "benchmark_index": int(meta.benchmark_index),
                "row_idx": int(meta.row_idx),
                "split": str(meta.split),
                "question": str(meta.question),
                "target_category": str(meta.target_category),
                "clean_p_yes": float(meta.clean_p_yes),
                "patch_index": int(patch_index),
                "grid_x": int(patch.grid_x),
                "grid_y": int(patch.grid_y),
                "patch_box_json": str(patch.patch_box_json),
                "patch_has_target": int(patch.patch_has_target),
                "patch_target_overlap_frac": float(
                    patch.patch_target_overlap_frac
                ),
                "target_coverage_frac": float(
                    patch.target_coverage_frac
                ),
            })

    pending_heatmap = pd.DataFrame(task_rows)
    new_rows = []

    for start in tqdm(
        range(0, len(pending_heatmap), batch_size),
        desc=f"{tag} controlled heatmaps",
    ):
        batch = pending_heatmap.iloc[
            start:start + batch_size
        ]
        images = []
        prompts = []
        clean_hiddens = []

        for _, task in batch.iterrows():
            image = get_image(int(task.row_idx))
            box = json.loads(task.patch_box_json)
            images.append(
                apply_patch_blur(image, [box])
            )
            prompts.append(
                answer_prompt(task.question)
            )
            clean_hiddens.append(
                np.asarray(
                    np.load(
                        hidden_dir
                        / f"benchmark_{int(task.benchmark_index)}.npy"
                    ),
                    dtype=np.float32,
                )
            )

        output = qwen_batch(
            model,
            processor,
            images,
            prompts,
            return_hidden=True,
        )

        for index, (_, task) in enumerate(
            batch.iterrows()
        ):
            clean_hidden = clean_hiddens[index]
            perturbed_hidden = np.asarray(
                output["hidden"][index],
                dtype=np.float32,
            )
            difference = (
                perturbed_hidden[heatmap_layer]
                - clean_hidden[heatmap_layer]
            )
            relative_l2 = float(
                np.linalg.norm(difference)
                / (
                    np.linalg.norm(
                        clean_hidden[heatmap_layer]
                    ) + 1e-8
                )
            )

            new_rows.append({
                **task.to_dict(),
                "heatmap_layer": int(heatmap_layer),
                "relative_l2": relative_l2,
                "perturb_p_yes": float(
                    output["p_yes"][index]
                ),
                "p_yes_drop": float(
                    task.clean_p_yes
                    - output["p_yes"][index]
                ),
                "perturb_raw": output["raw"][index],
            })

        if (
            len(new_rows) >= 240
            or start + batch_size >= len(pending_heatmap)
        ):
            heatmap = pd.concat(
                [heatmap, pd.DataFrame(new_rows)],
                ignore_index=True,
            )
            heatmap = heatmap.drop_duplicates(
                "condition_id", keep="last"
            )
            heatmap.to_csv(heatmap_csv, index=False)
            new_rows = []

    # Build frozen hidden top k and random k necessity and sufficiency tasks.
    intervention_tasks_csv = (
        model_dir / "intervention_tasks.csv"
    )
    intervention_results_csv = (
        model_dir / "intervention_results.csv"
    )

    if intervention_tasks_csv.exists():
        intervention_tasks = pd.read_csv(
            intervention_tasks_csv
        )
    else:
        rows = []
        for benchmark_index, group in heatmap.groupby(
            "benchmark_index", sort=False
        ):
            group = group.sort_values(
                "relative_l2", ascending=False
            )
            all_indices = sorted(
                group.patch_index.astype(int).tolist()
            )
            box_by_index = {
                int(row.patch_index): str(row.patch_box_json)
                for _, row in group.iterrows()
            }
            meta = selected_model[
                selected_model.benchmark_index.astype(int)
                == int(benchmark_index)
            ].iloc[0]

            for k in K_VALUES:
                top_indices = (
                    group.patch_index.astype(int)
                    .tolist()[:k]
                )
                eligible = [
                    index for index in all_indices
                    if index not in top_indices
                ]
                rng = np.random.default_rng(
                    SEED
                    + int(benchmark_index) * 1009
                    + k * 101
                    + (0 if tag == "qwen7b" else 50000)
                )
                random_indices = rng.choice(
                    eligible,
                    size=k,
                    replace=False,
                ).astype(int).tolist()

                for selection_type, indices in [
                    ("top", top_indices),
                    ("random", random_indices),
                ]:
                    boxes = [
                        json.loads(box_by_index[index])
                        for index in indices
                    ]
                    for intervention in [
                        "necessity",
                        "sufficiency",
                    ]:
                        rows.append({
                            "condition_id": (
                                f"{int(benchmark_index)}|"
                                f"{k}|{selection_type}|"
                                f"{intervention}"
                            ),
                            "benchmark_index": int(
                                benchmark_index
                            ),
                            "row_idx": int(meta.row_idx),
                            "split": str(meta.split),
                            "question": str(meta.question),
                            "target_category": str(
                                meta.target_category
                            ),
                            "clean_p_yes": float(
                                meta.clean_p_yes
                            ),
                            "k": int(k),
                            "selection_type": selection_type,
                            "intervention": intervention,
                            "patch_indices_json": json.dumps(
                                indices
                            ),
                            "patch_boxes_json": json.dumps(
                                boxes
                            ),
                        })

        intervention_tasks = pd.DataFrame(rows)
        intervention_tasks.to_csv(
            intervention_tasks_csv, index=False
        )

    if intervention_results_csv.exists():
        intervention_results = pd.read_csv(
            intervention_results_csv
        )
    else:
        intervention_results = pd.DataFrame()

    completed_interventions = set(
        intervention_results.get(
            "condition_id", pd.Series(dtype=str)
        ).astype(str).tolist()
    )
    pending_interventions = intervention_tasks[
        ~intervention_tasks.condition_id.astype(str).isin(
            completed_interventions
        )
    ].reset_index(drop=True)

    new_rows = []

    for start in tqdm(
        range(0, len(pending_interventions), batch_size),
        desc=f"{tag} necessity sufficiency",
    ):
        batch = pending_interventions.iloc[
            start:start + batch_size
        ]
        images = []
        prompts = []

        for _, task in batch.iterrows():
            image = get_image(int(task.row_idx))
            boxes = json.loads(task.patch_boxes_json)
            if str(task.intervention) == "necessity":
                edited = apply_patch_blur(image, boxes)
            else:
                edited = keep_only_boxes(image, boxes)
            images.append(edited)
            prompts.append(answer_prompt(task.question))

            image_path = (
                evidence_dir
                / (
                    str(task.condition_id).replace("|", "_")
                    + ".jpg"
                )
            )
            if not image_path.exists():
                edited.save(image_path, quality=92)

        output = qwen_batch(
            model,
            processor,
            images,
            prompts,
            return_hidden=False,
        )

        for index, (_, task) in enumerate(
            batch.iterrows()
        ):
            new_rows.append({
                **task.to_dict(),
                "edited_raw": output["raw"][index],
                "edited_p_yes": float(
                    output["p_yes"][index]
                ),
                "edited_pred_yes": int(
                    output["pred"][index]
                ),
                "p_yes_drop": float(
                    task.clean_p_yes
                    - output["p_yes"][index]
                ),
            })

        if (
            len(new_rows) >= 160
            or start + batch_size
            >= len(pending_interventions)
        ):
            intervention_results = pd.concat(
                [
                    intervention_results,
                    pd.DataFrame(new_rows),
                ],
                ignore_index=True,
            )
            intervention_results = (
                intervention_results.drop_duplicates(
                    "condition_id", keep="last"
                )
            )
            intervention_results.to_csv(
                intervention_results_csv,
                index=False,
            )
            new_rows = []

    del model, processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(
        tag,
        "complete:",
        len(clean),
        "clean,",
        len(heatmap),
        "heatmap rows,",
        len(intervention_results),
        "interventions",
    )

print("Runner defined. Continue to Cells 6 and 7.")


In [ ]:
# Cell 6: run the controlled benchmark on Qwen2.5 VL 7B

if RUN_CONTROLLED_BENCHMARK:
    run_controlled_model(MODEL_SPECS[0])
else:
    print("RUN_CONTROLLED_BENCHMARK is False")


In [ ]:
# Cell 7: run the controlled benchmark on Qwen2.5 VL 3B

if RUN_CONTROLLED_BENCHMARK:
    run_controlled_model(MODEL_SPECS[1])
else:
    print("RUN_CONTROLLED_BENCHMARK is False")


In [ ]:
# Cell 8: aggregate controlled benchmark localization, necessity, and sufficiency

controlled_metric_rows = []
controlled_per_image_rows = []
curve_rows = []

benchmark_patch_base = benchmark_patches.copy()
if RUN_CLIP_BASELINE and len(clip_patch_scores):
    benchmark_patch_base = benchmark_patch_base.merge(
        clip_patch_scores,
        on=["benchmark_index", "patch_index"],
        how="left",
    )
else:
    benchmark_patch_base["clip_score"] = 0.0

for spec in MODEL_SPECS:
    tag = spec["tag"]
    model_dir = CONTROL_DIR / tag

    clean = pd.read_csv(
        model_dir / "clean_predictions.csv"
    )
    heatmap = pd.read_csv(
        model_dir / "heatmap_results.csv"
    )
    interventions = pd.read_csv(
        model_dir / "intervention_results.csv"
    )

    eval_indices = set(
        clean[
            (clean.split.astype(str) == "eval")
            & (clean.clean_pred.astype(int) == 1)
        ].benchmark_index.astype(int).tolist()
    )
    heatmap_eval = heatmap[
        heatmap.benchmark_index.astype(int).isin(
            eval_indices
        )
    ].copy()

    combined = heatmap_eval.merge(
        benchmark_patch_base[[
            "benchmark_index",
            "patch_index",
            "patch_box_json",
            "patch_has_target",
            "patch_target_overlap_frac",
            "target_coverage_frac",
            "clip_score",
        ]],
        on=["benchmark_index", "patch_index"],
        how="left",
        suffixes=("", "_base"),
    )

    parts = []
    for benchmark_index, group in combined.groupby(
        "benchmark_index", sort=False
    ):
        group = group.copy().sort_values("patch_index")
        meta = clean[
            clean.benchmark_index.astype(int)
            == int(benchmark_index)
        ].iloc[0]
        image = get_image(int(meta.row_idx))
        group["center_score"] = center_score_for_boxes(
            group.patch_box_json.astype(str).tolist(),
            image.size,
        )
        rng = np.random.default_rng(
            SEED
            + int(benchmark_index) * 307
            + (0 if tag == "qwen7b" else 90000)
        )
        group["random_score"] = rng.random(len(group))
        group["oracle_score"] = pd.to_numeric(
            group.patch_target_overlap_frac,
            errors="coerce",
        ).fillna(0.0)
        parts.append(group)

    combined = pd.concat(parts, ignore_index=True)

    methods = {
        "hidden_state": "relative_l2",
        "output_drop": "p_yes_drop",
        "clip": "clip_score",
        "center": "center_score",
        "random": "random_score",
        "target_oracle": "oracle_score",
    }

    for benchmark_index, group in combined.groupby(
        "benchmark_index", sort=False
    ):
        for method, column in methods.items():
            metrics = localization_metrics_for_group(
                group, column
            )
            controlled_per_image_rows.append({
                "model": tag,
                "benchmark_index": int(benchmark_index),
                "method": method,
                **metrics,
            })

    eval_interventions = interventions[
        interventions.benchmark_index.astype(int).isin(
            eval_indices
        )
    ].copy()

    summary = eval_interventions.groupby(
        ["k", "selection_type", "intervention"],
        as_index=False,
    ).agg(
        mean_edited_p_yes=("edited_p_yes", "mean"),
        mean_p_yes_drop=("p_yes_drop", "mean"),
        answer_yes_rate=("edited_pred_yes", "mean"),
        n=("condition_id", "count"),
    )
    summary["model"] = tag
    curve_rows.extend(summary.to_dict(orient="records"))

controlled_per_image = pd.DataFrame(
    controlled_per_image_rows
)

for (model_tag, method), group in controlled_per_image.groupby(
    ["model", "method"], sort=False
):
    row = {
        "model": model_tag,
        "method": method,
        "n_images": int(len(group)),
    }
    for column in [
        "patch_ap",
        "reciprocal_first_target_rank",
        "top1_hit",
        "top1_coverage",
        "top3_hit",
        "top3_coverage",
        "top5_hit",
        "top5_coverage",
    ]:
        summary = bootstrap_mean(
            group[column].to_numpy(float),
            seed=SEED + abs(hash((model_tag, method, column))) % 10000,
        )
        row[column] = summary["mean"]
        row[f"{column}_ci_low"] = summary["ci_low"]
        row[f"{column}_ci_high"] = summary["ci_high"]
    controlled_metric_rows.append(row)

controlled_summary = pd.DataFrame(
    controlled_metric_rows
)
controlled_curves = pd.DataFrame(curve_rows)

controlled_summary.to_csv(
    CONTROL_DIR / "controlled_localization_summary.csv",
    index=False,
)
controlled_per_image.to_csv(
    CONTROL_DIR / "controlled_localization_per_image.csv",
    index=False,
)
controlled_curves.to_csv(
    CONTROL_DIR / "controlled_necessity_sufficiency_curves.csv",
    index=False,
)

print("\nControlled localization")
print(controlled_summary[[
    "model",
    "method",
    "patch_ap",
    "top1_hit",
    "top3_coverage",
    "top5_coverage",
]])

print("\nControlled necessity and sufficiency")
print(controlled_curves)

# ============================================================
# Plots
# ============================================================

import matplotlib.pyplot as plt

hidden_summary = controlled_summary[
    controlled_summary.method == "hidden_state"
].copy()
baseline_summary = controlled_summary[
    controlled_summary.method.isin([
        "output_drop", "clip", "center", "random"
    ])
].copy()

fig, axis = plt.subplots(figsize=(8.5, 4.8))
pivot = controlled_summary.pivot(
    index="method",
    columns="model",
    values="patch_ap",
)
pivot.plot(kind="bar", ax=axis)
axis.set_ylabel("mean patch average precision")
axis.set_title("Controlled COCO target localization")
axis.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "controlled_localization_baselines.png",
    dpi=240,
)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.3))
for axis, intervention in zip(
    axes, ["necessity", "sufficiency"]
):
    subset = controlled_curves[
        controlled_curves.intervention == intervention
    ]
    for (model_tag, selection_type), group in subset.groupby(
        ["model", "selection_type"]
    ):
        group = group.sort_values("k")
        y_column = (
            "mean_p_yes_drop"
            if intervention == "necessity"
            else "mean_edited_p_yes"
        )
        axis.plot(
            group.k,
            group[y_column],
            marker="o",
            label=f"{model_tag}, {selection_type}",
        )
    axis.set_title(intervention)
    axis.set_xlabel("k patches")
    axis.set_xticks(K_VALUES)
axes[0].set_ylabel("probability effect")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "controlled_necessity_sufficiency_curves.png",
    dpi=240,
)
plt.close(fig)

print("Cell 8 complete.")


In [ ]:
# Cell 9: localization versus verification gap and selective verification analysis

GAP_DIR = OUT_DIR / "localization_verification_gap"
GAP_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Verification metrics from completed matched experiments
# ============================================================

verification_rows = []

verification_rows.append({
    "model": "qwen7b",
    "experiment": "7B discovery sufficiency",
    "verification_auroc": float(
        semantic_result_7b[
            "validation_selected_eval"
        ]["auroc"]
    ),
    "verification_auprc": float(
        semantic_result_7b[
            "validation_selected_eval"
        ]["auprc"]
    ),
    "verification_balanced_accuracy": float(
        semantic_result_7b[
            "validation_selected_eval"
        ]["balanced_accuracy"]
    ),
})

verification_rows.append({
    "model": "qwen3b",
    "experiment": "3B frozen transfer",
    "verification_auroc": float(
        result_3b["pooled_eval"]["auroc"]
    ),
    "verification_auprc": float(
        result_3b["pooled_eval"]["auprc"]
    ),
    "verification_balanced_accuracy": float(
        result_3b["pooled_eval"]["balanced_accuracy"]
    ),
})

verification_summary = pd.DataFrame(
    verification_rows
)

# Strongest controlled localization result for each model.
hidden_controlled = controlled_summary[
    controlled_summary.method == "hidden_state"
][[
    "model",
    "patch_ap",
    "top1_hit",
    "top3_coverage",
    "top5_coverage",
]].copy()

gap_summary = hidden_controlled.merge(
    verification_summary,
    on="model",
    how="left",
)
gap_summary["localization_minus_verification"] = (
    gap_summary.patch_ap
    - gap_summary.verification_auroc
)

gap_summary.to_csv(
    GAP_DIR / "localization_verification_gap_summary.csv",
    index=False,
)

print(gap_summary)

# ============================================================
# Heatmap concentration and coverage-risk analysis
# ============================================================

def concentration_table(frame, id_column, score_column):
    rows = []
    for identifier, group in frame.groupby(
        id_column, sort=False
    ):
        scores = pd.to_numeric(
            group[score_column],
            errors="coerce",
        ).fillna(0.0).to_numpy(float)
        scores = np.maximum(scores, 0.0)
        total = scores.sum()
        normalized = (
            scores / total
            if total > 0
            else np.full(len(scores), 1.0 / len(scores))
        )
        rows.append({
            id_column: int(identifier),
            "heatmap_entropy": normalized_entropy(scores),
            "heatmap_concentration": float(
                1.0 - normalized_entropy(scores)
            ),
            "top1_mass": float(
                np.sort(normalized)[-1]
            ),
            "top3_mass": float(
                np.sort(normalized)[-3:].sum()
            ),
        })
    return pd.DataFrame(rows)

concentration_7b = concentration_table(
    dist_7b[
        pd.to_numeric(
            dist_7b["layer"], errors="coerce"
        ).astype(int) == 21
    ],
    "selected_index",
    "relative_l2",
)

concentration_3b = concentration_table(
    heat_3b,
    "selected_index",
    "relative_l2",
)

selective_7b = semantic_all_7b.merge(
    concentration_7b,
    on="selected_index",
    how="left",
)
selective_7b["is_grounded"] = (
    selective_7b.clean_group.astype(str)
    == "grounded_yes"
).astype(int)
selective_7b["verification_score"] = (
    selective_7b.sufficiency_lift
)

selective_3b = analysis_3b.merge(
    concentration_3b,
    on="selected_index",
    how="left",
)
selective_3b["is_grounded"] = (
    selective_3b.clean_group.astype(str)
    == "grounded_yes"
).astype(int)
selective_3b["verification_score"] = (
    selective_3b.frozen_score
)

risk_rows = []

for model_tag, frame in [
    ("qwen7b", selective_7b),
    ("qwen3b", selective_3b),
]:
    eval_frame = frame[
        frame.split.astype(str) == "eval"
    ].dropna(
        subset=["verification_score", "heatmap_concentration"]
    ).copy()

    for quantile in [0.0, 0.25, 0.5, 0.75]:
        threshold = float(
            eval_frame.heatmap_concentration.quantile(
                quantile
            )
        )
        accepted = eval_frame[
            eval_frame.heatmap_concentration >= threshold
        ]
        if (
            len(accepted) < 8
            or len(accepted.is_grounded.unique()) < 2
        ):
            continue
        risk_rows.append({
            "model": model_tag,
            "minimum_concentration_quantile": quantile,
            "coverage": float(
                len(accepted) / len(eval_frame)
            ),
            "n": int(len(accepted)),
            "verification_auroc": float(
                roc_auc_score(
                    accepted.is_grounded,
                    accepted.verification_score,
                )
            ),
        })

risk_coverage = pd.DataFrame(risk_rows)
risk_coverage.to_csv(
    GAP_DIR / "selective_verification_risk_coverage.csv",
    index=False,
)

# ============================================================
# Gap figure
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.3))

x = np.arange(len(gap_summary))
width = 0.34
axes[0].bar(
    x - width / 2,
    gap_summary.patch_ap,
    width,
    label="causal localization AP",
)
axes[0].bar(
    x + width / 2,
    gap_summary.verification_auroc,
    width,
    label="verification AUROC",
)
axes[0].set_xticks(x)
axes[0].set_xticklabels(gap_summary.model)
axes[0].set_ylim(0, 1)
axes[0].set_title("Localization does not imply verification")
axes[0].legend(fontsize=8)

for model_tag, group in risk_coverage.groupby("model"):
    group = group.sort_values("coverage")
    axes[1].plot(
        group.coverage,
        group.verification_auroc,
        marker="o",
        label=model_tag,
    )
axes[1].axhline(0.5, linestyle=":")
axes[1].set_xlabel("coverage")
axes[1].set_ylabel("verification AUROC")
axes[1].set_title("Selective verification by heatmap concentration")
axes[1].legend()

fig.tight_layout()
fig.savefig(
    FIG_DIR / "localization_verification_gap.png",
    dpi=240,
)
plt.close(fig)

print("Cell 9 complete.")


In [ ]:
# Cell 10: hallucination mechanism taxonomy audit and blind labeling sheet

if RUN_TAXONOMY_AUDIT:
    taxonomy_rows = []

    # --------------------------------------------------------
    # 7B hallucinations
    # --------------------------------------------------------

    heat_layer_7b = dist_7b[
        pd.to_numeric(
            dist_7b["layer"], errors="coerce"
        ).astype(int) == 21
    ].copy()

    tasks_7b_primary = tasks_7b[
        (tasks_7b.selection_type.astype(str) == "topk")
        & (pd.to_numeric(tasks_7b.k, errors="coerce").astype(int) == 3)
    ][[
        "selected_index",
        "patch_indices_json",
    ]].drop_duplicates("selected_index")

    random_7b_primary = tasks_7b[
        (tasks_7b.selection_type.astype(str) == "random")
        & (pd.to_numeric(tasks_7b.k, errors="coerce").astype(int) == 3)
        & (
            pd.to_numeric(
                tasks_7b.random_trial, errors="coerce"
            ).fillna(0).astype(int) == 0
        )
    ][[
        "selected_index",
        "patch_indices_json",
    ]].drop_duplicates("selected_index")

    necessity_7b = conditions_7b[
        (conditions_7b.selection_type.astype(str) == "topk")
        & (conditions_7b.mask_mode.astype(str) == "blur")
        & (pd.to_numeric(conditions_7b.k, errors="coerce").astype(int) == 3)
    ][[
        "selected_index",
        "p_yes_drop",
    ]].rename(
        columns={"p_yes_drop": "necessity_drop"}
    )

    eval_hall_7b = semantic_all_7b[
        (semantic_all_7b.split.astype(str) == "eval")
        & (
            semantic_all_7b.clean_group.astype(str)
            == "hallucinated_yes"
        )
    ].copy()

    eval_hall_7b = (
        eval_hall_7b.merge(
            concentration_7b,
            on="selected_index",
            how="left",
        )
        .merge(
            necessity_7b,
            on="selected_index",
            how="left",
        )
        .merge(
            tasks_7b_primary,
            on="selected_index",
            how="left",
        )
        .merge(
            random_7b_primary,
            on="selected_index",
            how="left",
            suffixes=("_top", "_random"),
        )
    )

    patch_box_7b = {
        (
            int(row.selected_index),
            int(row.patch_index),
        ): [
            int(value)
            for value in json.loads(row.patch_box_json)
        ]
        for _, row in grid_7b.drop_duplicates(
            ["selected_index", "patch_index"]
        ).iterrows()
    }

    # --------------------------------------------------------
    # 3B hallucinations
    # --------------------------------------------------------

    task_top_3b = tasks_3b[
        tasks_3b.selection_type.astype(str) == "top3"
    ][[
        "selected_index",
        "patch_indices_json",
    ]].drop_duplicates("selected_index")

    task_random_3b = tasks_3b[
        tasks_3b.selection_type.astype(str) == "random3"
    ][[
        "selected_index",
        "patch_indices_json",
    ]].drop_duplicates("selected_index")

    eval_hall_3b = analysis_3b[
        (analysis_3b.split.astype(str) == "eval")
        & (
            analysis_3b.clean_group.astype(str)
            == "hallucinated_yes"
        )
    ].copy()

    eval_hall_3b = (
        eval_hall_3b.merge(
            concentration_3b,
            on="selected_index",
            how="left",
        )
        .merge(
            task_top_3b,
            on="selected_index",
            how="left",
        )
        .merge(
            task_random_3b,
            on="selected_index",
            how="left",
            suffixes=("_top", "_random"),
        )
    )

    patch_box_3b = {
        (
            int(row.selected_index),
            int(row.patch_index),
        ): [
            int(value)
            for value in json.loads(row.patch_box_json)
        ]
        for _, row in heat_3b.drop_duplicates(
            ["selected_index", "patch_index"]
        ).iterrows()
    }

    def auto_taxonomy_suggestion(
        entropy,
        top1_mass,
        necessity_drop,
        sufficiency_score,
        full_clip=None,
        inventory_target=None,
    ):
        if (
            full_clip is not None
            and np.isfinite(full_clip)
            and full_clip >= 0.9
        ) or (
            inventory_target is not None
            and int(inventory_target) == 1
        ):
            return "semantic_or_annotation_ambiguity_candidate"

        if (
            entropy <= 0.90
            and (
                (np.isfinite(necessity_drop) and necessity_drop >= 0.30)
                or top1_mass >= 0.18
            )
        ):
            return "localized_visual_confusion"

        if (
            entropy >= 0.96
            and (
                not np.isfinite(necessity_drop)
                or necessity_drop <= 0.20
            )
        ):
            return "diffuse_scene_prior"

        return "unexplained"

    html_parts = ["""
    <html>
    <head>
    <meta charset="utf-8">
    <style>
    body{font-family:Arial;margin:24px;background:#f5f5f5}
    .case{background:white;border:1px solid #ccc;border-radius:10px;padding:14px;margin:18px 0}
    .row{display:flex;gap:12px;flex-wrap:wrap}
    .panel{width:23%;min-width:220px}
    img{width:100%;border:1px solid #aaa}
    pre{white-space:pre-wrap;background:#eee;padding:8px}
    </style>
    </head>
    <body>
    <h1>Blind Hallucination Mechanism Audit</h1>
    <p>Fill the human_label column in taxonomy_labels.csv without using the automatic suggestion as ground truth.</p>
    """]

    def render_case(
        model_tag,
        meta,
        top_indices,
        random_indices,
        patch_lookup,
        heat_frame,
        score_column,
        score_value,
        necessity_drop,
        full_clip,
        inventory_target,
    ):
        image = get_image(int(meta.row_idx))
        selected_index = int(meta.selected_index)

        top_boxes = [
            patch_lookup[(selected_index, int(index))]
            for index in top_indices
        ]
        random_boxes = [
            patch_lookup[(selected_index, int(index))]
            for index in random_indices
        ]

        top_only = keep_only_boxes(image, top_boxes)
        random_only = keep_only_boxes(image, random_boxes)

        heat_rows = heat_frame[
            heat_frame.selected_index.astype(int)
            == selected_index
        ].copy()
        heat = heat_rows.pivot(
            index="grid_y",
            columns="grid_x",
            values=score_column,
        ).sort_index().sort_index(axis=1).to_numpy(float)

        overlay = image.copy()
        draw = ImageDraw.Draw(overlay)
        for box in top_boxes:
            draw.rectangle(
                box[:4], outline=(255, 0, 0), width=3
            )

        heat_image = image.copy()
        # Save a simple overlay via matplotlib.
        import matplotlib.pyplot as plt
        figure, axis = plt.subplots(figsize=(4, 4))
        axis.imshow(image)
        axis.imshow(
            heat,
            alpha=0.52,
            extent=[0, image.width, image.height, 0],
            interpolation="bilinear",
        )
        axis.axis("off")
        buffer = io.BytesIO()
        figure.savefig(
            buffer,
            format="png",
            dpi=150,
            bbox_inches="tight",
            pad_inches=0,
        )
        plt.close(figure)
        heat_b64 = base64.b64encode(
            buffer.getvalue()
        ).decode("utf-8")

        entropy = float(meta.heatmap_entropy)
        top1_mass = float(meta.top1_mass)
        suggestion = auto_taxonomy_suggestion(
            entropy,
            top1_mass,
            float(necessity_drop),
            float(score_value),
            full_clip,
            inventory_target,
        )

        case_id = (
            f"{model_tag}_{selected_index}"
        )
        taxonomy_rows.append({
            "case_id": case_id,
            "model": model_tag,
            "selected_index": selected_index,
            "row_idx": int(meta.row_idx),
            "question": str(meta.question),
            "target_category": str(meta.target_category),
            "pope_category": str(
                getattr(meta, "pope_category", getattr(meta, "pair_category", "unknown"))
            ),
            "clean_p_yes": float(meta.clean_p_yes),
            "heatmap_entropy": entropy,
            "top1_mass": top1_mass,
            "necessity_drop": float(necessity_drop),
            "sufficiency_score": float(score_value),
            "auto_suggestion": suggestion,
            "human_label": "",
            "human_notes": "",
        })

        html_parts.append(
            f"""
            <div class="case">
            <h2>{html.escape(case_id)}</h2>
            <pre>{html.escape(str(meta.question))}
target: {html.escape(str(meta.target_category))}
model: {model_tag}
clean p_yes: {float(meta.clean_p_yes):.3f}
heatmap entropy: {entropy:.3f}
top1 mass: {top1_mass:.3f}
necessity drop: {float(necessity_drop):.3f}
sufficiency score: {float(score_value):.3f}
auto suggestion: {html.escape(suggestion)}</pre>
            <div class="row">
              <div class="panel"><h3>original</h3><img src="data:image/jpeg;base64,{image_to_base64(image)}"></div>
              <div class="panel"><h3>heatmap</h3><img src="data:image/png;base64,{heat_b64}"></div>
              <div class="panel"><h3>top 3 only</h3><img src="data:image/jpeg;base64,{image_to_base64(top_only)}"></div>
              <div class="panel"><h3>random 3 only</h3><img src="data:image/jpeg;base64,{image_to_base64(random_only)}"></div>
            </div>
            </div>
            """
        )

    for _, meta in tqdm(
        eval_hall_7b.iterrows(),
        total=len(eval_hall_7b),
        desc="7B taxonomy audit",
    ):
        top_indices = [
            int(value)
            for value in json.loads(
                meta.patch_indices_json_top
            )
        ]
        random_indices = [
            int(value)
            for value in json.loads(
                meta.patch_indices_json_random
            )
        ]
        render_case(
            "qwen7b",
            meta,
            top_indices,
            random_indices,
            patch_box_7b,
            dist_7b[
                pd.to_numeric(
                    dist_7b["layer"], errors="coerce"
                ).astype(int) == 21
            ],
            "relative_l2",
            float(meta.sufficiency_lift),
            float(meta.necessity_drop),
            float(
                getattr(meta, "semantic_clip_full", np.nan)
            ),
            getattr(meta, "inventory_full", np.nan),
        )

    for _, meta in tqdm(
        eval_hall_3b.iterrows(),
        total=len(eval_hall_3b),
        desc="3B taxonomy audit",
    ):
        top_indices = [
            int(value)
            for value in json.loads(
                meta.patch_indices_json_top
            )
        ]
        random_indices = [
            int(value)
            for value in json.loads(
                meta.patch_indices_json_random
            )
        ]
        render_case(
            "qwen3b",
            meta,
            top_indices,
            random_indices,
            patch_box_3b,
            heat_3b,
            "relative_l2",
            float(meta.frozen_score),
            float("nan"),
            None,
            None,
        )

    html_parts.append("</body></html>")

    taxonomy_labels = pd.DataFrame(
        taxonomy_rows
    )
    taxonomy_labels.to_csv(
        AUDIT_DIR / "taxonomy_labels.csv",
        index=False,
    )
    (
        AUDIT_DIR / "taxonomy_audit.html"
    ).write_text(
        "\n".join(html_parts),
        encoding="utf-8",
    )

    provisional_summary = (
        taxonomy_labels.groupby(
            ["model", "auto_suggestion"]
        )
        .size()
        .reset_index(name="n")
    )
    provisional_summary.to_csv(
        AUDIT_DIR / "provisional_taxonomy_summary.csv",
        index=False,
    )

    print(provisional_summary)
    print(
        "Human labeling sheet:",
        AUDIT_DIR / "taxonomy_labels.csv",
    )
else:
    print("RUN_TAXONOMY_AUDIT is False")

print("Cell 10 complete.")


In [ ]:
# Cell 11: final paper package, figures, macros, and export zip

FINAL_DIR = OUT_DIR / "final_paper_package"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Core numerical summary
# ============================================================

hidden_control = controlled_summary[
    controlled_summary.method == "hidden_state"
].set_index("model")

random_control = controlled_summary[
    controlled_summary.method == "random"
].set_index("model")

final_summary = {
    "title": (
        "Self Verification Through Hidden Visual Evidence "
        "in Vision Language Models"
    ),
    "central_claim": (
        "Hidden-state perturbations can localize causal visual "
        "evidence across models, but causal localization does not "
        "reliably imply that the evidence semantically justifies "
        "the answer."
    ),
    "controlled_localization": {
        model_tag: {
            "hidden_patch_ap": float(
                hidden_control.loc[model_tag, "patch_ap"]
            ),
            "hidden_top1_hit": float(
                hidden_control.loc[model_tag, "top1_hit"]
            ),
            "hidden_top3_coverage": float(
                hidden_control.loc[model_tag, "top3_coverage"]
            ),
            "random_patch_ap": float(
                random_control.loc[model_tag, "patch_ap"]
            ),
            "random_top1_hit": float(
                random_control.loc[model_tag, "top1_hit"]
            ),
        }
        for model_tag in ["qwen7b", "qwen3b"]
    },
    "verification": {
        row.model: {
            "experiment": row.experiment,
            "auroc": float(row.verification_auroc),
            "auprc": float(row.verification_auprc),
            "balanced_accuracy": float(
                row.verification_balanced_accuracy
            ),
        }
        for _, row in verification_summary.iterrows()
    },
    "taxonomy_requires_human_labels": True,
}

(
    FINAL_DIR / "final_results_summary.json"
).write_text(json.dumps(final_summary, indent=2))

summary_text = f"""
TITLE
{final_summary['title']}

CENTRAL RESULT
{final_summary['central_claim']}

CONTROLLED CAUSAL LOCALIZATION
Qwen 7B hidden patch AP: {hidden_control.loc['qwen7b', 'patch_ap']:.3f}
Qwen 7B hidden top 1 hit: {hidden_control.loc['qwen7b', 'top1_hit']:.3f}
Qwen 7B random patch AP: {random_control.loc['qwen7b', 'patch_ap']:.3f}

Qwen 3B hidden patch AP: {hidden_control.loc['qwen3b', 'patch_ap']:.3f}
Qwen 3B hidden top 1 hit: {hidden_control.loc['qwen3b', 'top1_hit']:.3f}
Qwen 3B random patch AP: {random_control.loc['qwen3b', 'patch_ap']:.3f}

VERIFICATION
Qwen 7B verification AUROC: {verification_summary[verification_summary.model == 'qwen7b'].iloc[0].verification_auroc:.3f}
Qwen 3B frozen transfer AUROC: {verification_summary[verification_summary.model == 'qwen3b'].iloc[0].verification_auroc:.3f}

INTERPRETATION
Causal visual attribution transfers better than hallucination verification.
Necessity and sufficiency measure dependence on selected evidence, not whether that evidence is semantically valid.
"""
(
    FINAL_DIR / "final_results_summary.txt"
).write_text(summary_text.strip() + "\n")

# ============================================================
# Main paper figure
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.5))

# Panel A: localization AP.
plot_data = controlled_summary[
    controlled_summary.method.isin([
        "hidden_state", "output_drop", "clip", "center", "random"
    ])
].copy()
for model_index, model_tag in enumerate(["qwen7b", "qwen3b"]):
    group = plot_data[
        plot_data.model == model_tag
    ].set_index("method")
    methods = [
        "hidden_state", "output_drop", "clip", "center", "random"
    ]
    values = [float(group.loc[method, "patch_ap"]) for method in methods]
    x = np.arange(len(methods)) + model_index * 0.36
    axes[0, 0].bar(
        x,
        values,
        width=0.34,
        label=model_tag,
    )
axes[0, 0].set_xticks(
    np.arange(5) + 0.18
)
axes[0, 0].set_xticklabels(
    ["hidden", "output", "CLIP", "center", "random"],
    rotation=25,
)
axes[0, 0].set_ylabel("patch AP")
axes[0, 0].set_title("A. Causal target localization")
axes[0, 0].legend(fontsize=8)

# Panel B: top 1 hit.
for model_index, model_tag in enumerate(["qwen7b", "qwen3b"]):
    group = plot_data[
        plot_data.model == model_tag
    ].set_index("method")
    methods = [
        "hidden_state", "output_drop", "clip", "center", "random"
    ]
    values = [float(group.loc[method, "top1_hit"]) for method in methods]
    x = np.arange(len(methods)) + model_index * 0.36
    axes[0, 1].bar(
        x,
        values,
        width=0.34,
        label=model_tag,
    )
axes[0, 1].set_xticks(
    np.arange(5) + 0.18
)
axes[0, 1].set_xticklabels(
    ["hidden", "output", "CLIP", "center", "random"],
    rotation=25,
)
axes[0, 1].set_ylabel("top 1 target hit")
axes[0, 1].set_title("B. First selected region hits target")

# Panel C: necessity and sufficiency.
for model_tag in ["qwen7b", "qwen3b"]:
    model_curves = controlled_curves[
        controlled_curves.model == model_tag
    ]
    top_necessity = model_curves[
        (model_curves.selection_type == "top")
        & (model_curves.intervention == "necessity")
    ].sort_values("k")
    random_necessity = model_curves[
        (model_curves.selection_type == "random")
        & (model_curves.intervention == "necessity")
    ].sort_values("k")
    axes[1, 0].plot(
        top_necessity.k,
        top_necessity.mean_p_yes_drop
        - random_necessity.mean_p_yes_drop.to_numpy(),
        marker="o",
        label=model_tag,
    )
axes[1, 0].axhline(0, linestyle=":")
axes[1, 0].set_xticks(K_VALUES)
axes[1, 0].set_xlabel("k patches")
axes[1, 0].set_ylabel("necessity lift over random")
axes[1, 0].set_title("C. Selected evidence is causally necessary")
axes[1, 0].legend(fontsize=8)

# Panel D: localization verification gap.
x = np.arange(len(gap_summary))
width = 0.34
axes[1, 1].bar(
    x - width / 2,
    gap_summary.patch_ap,
    width,
    label="localization AP",
)
axes[1, 1].bar(
    x + width / 2,
    gap_summary.verification_auroc,
    width,
    label="verification AUROC",
)
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(gap_summary.model)
axes[1, 1].set_ylim(0, 1)
axes[1, 1].set_title("D. Localization does not imply verification")
axes[1, 1].legend(fontsize=8)

fig.suptitle(
    "Hidden visual evidence localizes causal regions, but does not guarantee valid self verification",
    fontsize=14,
)
fig.tight_layout(rect=[0, 0, 1, 0.96])
main_figure_path = (
    FINAL_DIR / "main_paper_figure.png"
)
fig.savefig(
    main_figure_path,
    dpi=260,
    bbox_inches="tight",
)
plt.close(fig)

# ============================================================
# LaTeX macros
# ============================================================

macros = f"""
\\newcommand{{\\SevenBLocalizationAP}}{{{hidden_control.loc['qwen7b', 'patch_ap']:.3f}}}
\\newcommand{{\\SevenBTopOneHit}}{{{hidden_control.loc['qwen7b', 'top1_hit']:.3f}}}
\\newcommand{{\\SevenBTopThreeCoverage}}{{{hidden_control.loc['qwen7b', 'top3_coverage']:.3f}}}

\\newcommand{{\\ThreeBLocalizationAP}}{{{hidden_control.loc['qwen3b', 'patch_ap']:.3f}}}
\\newcommand{{\\ThreeBTopOneHit}}{{{hidden_control.loc['qwen3b', 'top1_hit']:.3f}}}
\\newcommand{{\\ThreeBTopThreeCoverage}}{{{hidden_control.loc['qwen3b', 'top3_coverage']:.3f}}}

\\newcommand{{\\SevenBVerificationAUROC}}{{{verification_summary[verification_summary.model == 'qwen7b'].iloc[0].verification_auroc:.3f}}}
\\newcommand{{\\ThreeBVerificationAUROC}}{{{verification_summary[verification_summary.model == 'qwen3b'].iloc[0].verification_auroc:.3f}}}
"""
(
    FINAL_DIR / "paper_results_macros.tex"
).write_text(macros.strip() + "\n")

# ============================================================
# Copy key artifacts
# ============================================================

key_files = [
    CONTROL_DIR / "controlled_localization_summary.csv",
    CONTROL_DIR / "controlled_localization_per_image.csv",
    CONTROL_DIR / "controlled_necessity_sufficiency_curves.csv",
    GAP_DIR / "localization_verification_gap_summary.csv",
    GAP_DIR / "selective_verification_risk_coverage.csv",
    FIG_DIR / "controlled_localization_baselines.png",
    FIG_DIR / "controlled_necessity_sufficiency_curves.png",
    FIG_DIR / "localization_verification_gap.png",
]

for source in key_files:
    if source.exists():
        shutil.copy2(
            source,
            FINAL_DIR / source.name,
        )

if RUN_TAXONOMY_AUDIT:
    for source in AUDIT_DIR.iterdir():
        if source.is_file():
            shutil.copy2(
                source,
                FINAL_DIR / source.name,
            )

# ============================================================
# Export everything
# ============================================================

export_root = PROJECT_DIR / "export"
if export_root.exists():
    shutil.rmtree(export_root)
export_root.mkdir(parents=True, exist_ok=True)

# Include final package and compact analysis CSVs.
for root in [
    FINAL_DIR,
    LOCALIZATION_DIR,
    CONTROL_DIR,
    GAP_DIR,
    AUDIT_DIR,
]:
    if not root.exists():
        continue
    destination_root = export_root / root.name
    destination_root.mkdir(parents=True, exist_ok=True)
    for path in root.rglob("*"):
        if path.is_file():
            # Avoid copying hundreds of cached evidence images.
            if "evidence_images" in path.parts:
                continue
            destination = (
                destination_root
                / path.relative_to(root)
            )
            destination.parent.mkdir(
                parents=True, exist_ok=True
            )
            shutil.copy2(path, destination)

export_zip = shutil.make_archive(
    str(
        WORKSPACE
        / "pope_hidden_evidence_master_9of10_RESULTS"
    ),
    "zip",
    root_dir=export_root,
)

print("\nMASTER STUDY COMPLETE")
print("Main figure:", main_figure_path)
print("Final summary:", FINAL_DIR / "final_results_summary.txt")
print("Taxonomy sheet:", FINAL_DIR / "taxonomy_labels.csv")
print("Export zip:", export_zip)
